In [ ]:
# Task 1
#Random split 80/20 

In [1]:
# import packages 
import scanpy as sc
import pandas as pd

In [2]:
#Load the data
adata = sc.read_h5ad("../data/frangieh/adata_hvg_qc")

In [3]:
# Gene expression are the features
# Perturbation_2 are the treatment condition and is target variable
#Random split the qc done data to 80 /20 
#Train the random forest
#Evaluate via test set 

In [4]:
#Check the treatment condition
print(adata.obs["perturbation_2"].value_counts())

perturbation_2
IFNγ          74186
Co-culture    67093
Control       49895
Name: count, dtype: int64


In [5]:
# Train on highly variable genes
print(adata.var["highly_variable"].value_counts())

highly_variable
False    18711
True      5000
Name: count, dtype: int64


In [6]:
#Subset the data with only hvgs
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
print(adata_hvg.shape)

(191174, 5000)


In [7]:
#import packages
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [8]:
X = adata_hvg.X # imput features
y = adata_hvg.obs["perturbation_2"].values # target variables 

In [9]:
#Split the data now into test and train set 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42, # to make this split reproducable, it can be also 0, 1 etc. 
    stratify=y
)

In [ ]:
import gc
gc.collect()


In [12]:
import time  
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf = RandomForestClassifier(
    n_estimators=100,        # number of trees
    max_depth=15,            # maximum depth of each tree
    min_samples_split=5,     # minimum samples needed to split a node
    min_samples_leaf=2,      # minimum samples in a leaf
    max_features="sqrt",     # number of genes considered at each split
    class_weight="balanced", # useful if perturbation groups are imbalanced
    random_state=42,
    n_jobs=1
)

start = time.time()

rf.fit(X_train, y_train)

end = time.time()

print(f"Training time: {(end - start) / 60:.2f} minutes") # test for the future how many minutes

y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Training time: 6.89 minutes
Accuracy: 0.9776905976199817
              precision    recall  f1-score   support

  Co-culture       1.00      0.98      0.99     13419
     Control       0.92      1.00      0.96      9979
        IFNγ       1.00      0.96      0.98     14837

    accuracy                           0.98     38235
   macro avg       0.97      0.98      0.98     38235
weighted avg       0.98      0.98      0.98     38235

